Imports

In [1]:
from GNN.gnn import GNN
from GNN.gnn_train import build_graph, generate_sample, train
import cobra
import torch
from torch_geometric.data import HeteroData
import torch.nn as nn
import numpy as np
from policy.env_embeds import MetabolicEnv_gnn_embeds
from policy.env import MetabolicEnv_no_gnn
from policy.env_sequential import MetabolicEnv_sequential
from policy.env_gnn import MetabolicEnv_gnn
from ga.ga import GeneticAlgorithm
from ga.ga_embeds import GeneticAlgorithm_embeds
from stable_baselines3 import PPO
from GNN.mlp import GeneMLP, train_mlp

Simple MLP for flux prediction

In [2]:
cell_model = cobra.io.load_model("textbook")
network = GeneMLP(len(cell_model.genes))
train_mlp(cell_model, network)

/Users/rayenezanina/Desktop/Project/metabolic strain engineering/GNN/mlp.py:50: UserWarning: Using a target size (torch.Size([2])) that is different to the input size (torch.Size([1, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = torch.nn.functional.mse_loss(pred, target)
/Users/rayenezanina/Desktop/Project/metabolic strain engineering/.venv/lib/python3.14/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Epoch: 0, Loss: 1.3297374248504639
Epoch: 100, Loss: 0.8365588188171387
Epoch: 200, Loss: 0.6987273097038269
Epoch: 300, Loss: 0.7376725673675537
Epoch: 400, Loss: 0.7587371468544006
Epoch: 499, Loss: 0.6438698172569275


GNN for flux prediction

In [3]:
cell_model = cobra.io.load_model("textbook")
gnn = GNN(cell_model)
data, rxn_to_idx = build_graph(cell_model)

train(cell_model, gnn, 'Biomass_Ecoli_core','EX_etoh_e')

  0%|          | 1/500 [00:00<03:28,  2.39it/s]

Epoch: 0, Loss: 1010.9129638671875


 20%|██        | 101/500 [00:42<02:37,  2.53it/s]

Epoch: 100, Loss: 0.7957537770271301


 40%|████      | 201/500 [01:23<02:16,  2.18it/s]

...

Epoch: 300, Loss: 0.8201404809951782


 80%|████████  | 401/500 [02:48<00:40,  2.45it/s]

Epoch: 400, Loss: 0.39259007573127747


100%|██████████| 500/500 [03:28<00:00,  2.40it/s]

Epoch: 499, Loss: 0.22987127304077148


Simple PPO

In [4]:

no_gnn_env = MetabolicEnv_no_gnn('Biomass_Ecoli_core','EX_etoh_e')

model = PPO(
    "MlpPolicy",    
    no_gnn_env,
    n_steps=256,
    verbose=1,
    seed=42
)

model.learn(total_timesteps=30000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1        |
|    ep_rew_mean     | -100     |
| time/              |          |
|    fps             | 107      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 256      |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 1          |
|    ep_rew_mean          | -100       |
| time/                   |            |
|    fps                  | 108        |
|    iterations           | 2          |
|    time_elapsed         | 4          |
|    total_timesteps      | 512        |
| train/                  |            |
|    approx_kl            | 0.02284839 |
|    clip_fraction        | 0.11       |
|    clip_range           | 0.2        |
|    entropy_loss         | -95  

In [5]:
cell = cobra.io.load_model("textbook")
obs, info = no_gnn_env.reset()

action, _ = model.predict(obs, deterministic=False)
obs, reward, done, _, info = no_gnn_env.step(action)
genes = np.array([g.id for g in cell.genes])
print(genes[action.astype(bool)], reward, np.sum(action))

['b0351' 'b1276' 'b0474' 'b0727' 'b3735' 'b3732' 'b3738' 'b0733' 'b0979'
 'b3603' 'b3925' 'b0904' 'b4122' 'b1612' 'b3528' 'b0485' 'b3236' 'b1479'
 'b2463' 'b2286' 'b2288' 'b2277' 'b2285' 'b2278' 'b2283' 'b0451' 'b2579'
 'b0767' 'b4395' 'b1702' 'b0721'] 72.96100504118093 31.0


PPO with GNN embeds + predictions

In [6]:
gnn_env = MetabolicEnv_gnn(gnn,data,'Biomass_Ecoli_core','EX_etoh_e')

model = PPO(
    "MlpPolicy",    
    gnn_env,
    n_steps=256,
    verbose=1,
    seed=42
)

model.learn(total_timesteps=30000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1        |
|    ep_rew_mean     | -5.56    |
| time/              |          |
|    fps             | 276      |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 256      |
---------------------------------
---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 1         |
|    ep_rew_mean          | -5.29     |
| time/                   |           |
|    fps                  | 249       |
|    iterations           | 2         |
|    time_elapsed         | 2         |
|    total_timesteps      | 512       |
| train/                  |           |
|    approx_kl            | 0.2358768 |
|    clip_fraction        | 0.74      |
|    clip_range           | 0.2       |
|    entropy_loss         | -94.9     |
|    e

In [7]:
obs, info = gnn_env.reset()

action, _ = model.predict(obs, deterministic=False)
obs, reward, done, _, info = gnn_env.step(action)
print(genes[action.astype(bool)], reward, np.sum(action))

[] 1.31516415539653 0.0


PPO with GNN embeds

In [8]:
embed_env = MetabolicEnv_gnn_embeds(gnn,data,'Biomass_Ecoli_core','EX_etoh_e')

model = PPO(
    "MlpPolicy",    
    embed_env,
    n_steps=256,
    verbose=1,
    seed=42
)

model.learn(total_timesteps=30000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1        |
|    ep_rew_mean     | -100     |
| time/              |          |
|    fps             | 67       |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 256      |
---------------------------------
--------------------------------------
| rollout/                |          |
|    ep_len_mean          | 1        |
|    ep_rew_mean          | -100     |
| time/                   |          |
|    fps                  | 65       |
|    iterations           | 2        |
|    time_elapsed         | 7        |
|    total_timesteps      | 512      |
| train/                  |          |
|    approx_kl            | 0.0      |
|    clip_fraction        | 0        |
|    clip_range           | 0.2      |
|    entropy_loss         | -95      |
|    explained_varia

In [9]:
obs, info = embed_env.reset()

action, _ = model.predict(obs, deterministic=False)
obs, reward, done, _, info = embed_env.step(action)
print(genes[action.astype(bool)], reward, np.sum(action))

['b2296' 'b0727' 'b3732' 'b3731' 'b3737' 'b0979' 'b4152' 'b4153' 'b4122'
 'b1612' 'b1852' 'b0809' 'b0810' 'b1812' 'b0485' 'b3212' 'b2280' 'b0451'
 'b2579' 'b3386'] 74.06100518423204 20.0


Sequential PPO with GNN embeds

In [10]:
seq_env = MetabolicEnv_sequential(gnn,data,'Biomass_Ecoli_core','EX_etoh_e')

model = PPO(
    "MlpPolicy",    
    seq_env,
    n_steps=256,
    verbose=1,
    seed=42
)

model.learn(total_timesteps=50000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 4.58     |
|    ep_rew_mean     | 13.3     |
| time/              |          |
|    fps             | 143      |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 256      |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 4.63         |
|    ep_rew_mean          | 22.9         |
| time/                   |              |
|    fps                  | 158          |
|    iterations           | 2            |
|    time_elapsed         | 3            |
|    total_timesteps      | 512          |
| train/                  |              |
|    approx_kl            | 0.0137941735 |
|    clip_fraction        | 0.0707       |
|    clip_range           | 0.2          |
|    en

In [11]:
obs, info = seq_env.reset()
done = False
count = 0
while not done and count < 10:
    action, _ = model.predict(obs, deterministic=True)
    print(action)
    if np.ndim(action) == 0:
        action = int(action)
    obs, reward, done, _, info = seq_env.step(action)
    print(seq_env.current_knockouts, '\n', reward)
    count += 1

86
{'b2287'} 
 40.63501834520002
86
{'b2287'} 
 -100
86
{'b2287'} 
 -100
86
{'b2287'} 
 -100
86
{'b2287'} 
 -100
86
{'b2287'} 
 -100
86
{'b2287'} 
 -100
86
{'b2287'} 
 -100
86
{'b2287'} 
 -100
86
{'b2287'} 
 -100


Genetic algorithm

In [ ]:
cell_model = cobra.io.load_model("textbook")
ga = GeneticAlgorithm(cell_model, 'EX_etoh_e', 'Biomass_Ecoli_core')
np.random.seed(0)
best_knockouts, best_reward = ga.run()
print("Best knockouts:", best_knockouts)
print("Best reward:", best_reward)

/Users/rayenezanina/Desktop/Project/metabolic strain engineering/.venv/lib/python3.14/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Best knockouts: ['b3735', 'b2029', 'b2279', 'b0722']
Best reward: 75.66100518423156


Random search

In [5]:
env = MetabolicEnv_no_gnn('Biomass_Ecoli_core','EX_etoh_e')
best_reward = -1e6
best_knockouts = None
genes = list(env.model.genes)
for _ in range(30000):
    action = env.action_space.sample()
    _, reward, _, _, _ = env.step(action)
    if reward > best_reward:
        best_reward = reward
        best_knockouts = [genes[i].id for i in range(len(genes)) if action[i] == 1]
print("Best knockouts:", best_knockouts)
print("Best reward:", best_reward)

/Users/rayenezanina/Desktop/Project/metabolic strain engineering/.venv/lib/python3.14/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Best knockouts: ['b1241', 'b1276', 'b0474', 'b0116', 'b2587', 'b0356', 'b3735', 'b3733', 'b3736', 'b0979', 'b0978', 'b2975', 'b2779', 'b1773', 'b0904', 'b4154', 'b4153', 'b1819', 'b1818', 'b1611', 'b4122', 'b3528', 'b1101', 'b1297', 'b1761', 'b1812', 'b0485', 'b3213', 'b4077', 'b0875', 'b1136', 'b4015', 'b1479', 'b2463', 'b2286', 'b2279', 'b2288', 'b2287', 'b2281', 'b2277', 'b2285', 'b2282', 'b0451', 'b0114', 'b0115', 'b1723', 'b0902', 'b0903', 'b2926', 'b0767', 'b0755', 'b3956', 'b3403', 'b1702', 'b2297', 'b2458', 'b1676', 'b1854', 'b3386', 'b0722', 'b0724', 'b0723', 'b0008', 'b2935', 'b2465']
Best reward: -100


Genetic algorithm with GNN embeds for novelty

In [2]:
cell_model = cobra.io.load_model("textbook")
gnn = GNN(cell_model)
data, rxn_to_idx = build_graph(cell_model)

train(cell_model, gnn, 'Biomass_Ecoli_core','EX_etoh_e')

ga_embeds = GeneticAlgorithm_embeds(cell_model, gnn, data, 'EX_etoh_e', 'Biomass_Ecoli_core')
np.random.seed(0)
best_knockouts, best_reward = ga_embeds.run()
print("Best knockouts:", best_knockouts)
print("Best reward:", best_reward)

  0%|          | 0/300 [00:00<?, ?it/s]/Users/rayenezanina/Desktop/Project/metabolic strain engineering/.venv/lib/python3.14/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
  0%|          | 1/300 [00:00<02:21,  2.12it/s]

Epoch: 0, Loss: 4639.9658203125


 17%|█▋        | 51/300 [00:15<01:17,  3.23it/s]

Epoch: 50, Loss: 5.708690643310547


 34%|███▎      | 101/300 [00:30<01:01,  3.23it/s]

...

Epoch: 250, Loss: 4.567920207977295


100%|██████████| 300/300 [01:30<00:00,  3.30it/s]


Epoch: 299, Loss: 0.49892279505729675


100%|██████████| 200/200 [03:16<00:00,  1.02it/s]

Best knockouts: ['b3732', 'b2029', 'b2285', 'b0722']
Best reward: 75.66100518423156


using other models makes my kernel crash unfortunately

In [ ]:
new_model = cobra.io.load_model("iJR904")
new_ga = GeneticAlgorithm(new_model, 'EX_etoh_e', 'BIOMASS_Ecoli')

np.random.seed(0)
best_knockouts, best_reward = new_ga.run()
print("Best knockouts:", best_knockouts)
print("Best reward:", best_reward)

Output()

  0%|          | 0/200 [00:00<?, ?it/s]/Users/rayenezanina/Desktop/Project/metabolic strain engineering/.venv/lib/python3.14/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
100%|██████████| 200/200 [32:13<00:00,  9.67s/it]

Best knockouts: ['b0430', 'b2224', 'b2415', 'b1818', 'b3455', 'b0512', 'b2979', 'b3222', 'b2988', 'b0339', 'b0860', 'b0003', 'b3946', 'b2574', 'b1363', 'b3735', 'b1380', 'b2133', 'b3734', 'b4153', 'b3622', 'b3627', 'b0331', 'b0678', 'b2128', 'b2130', 'b0314', 'b2093', 'b2042', 'b3941', 'b0124', 'b1622', 'b0931', 'b0733', 'b1223', 'b1584', 'b3833', 'b0728', 'b2425', 'b4041', 'b0115', 'b0179', 'b3904', 'b4090', 'b2796']
Best reward: 27.578051904949483


: 

In [ ]:
cell_model = cobra.io.load_model("iJR904")
gnn = GNN(cell_model)
data, rxn_to_idx = build_graph(cell_model)

train(cell_model, gnn, 'BIOMASS_Ecoli','EX_etoh_e')

ga_embeds = GeneticAlgorithm_embeds(cell_model, gnn, data, 'EX_etoh_e', 'BIOMASS_Ecoli')
np.random.seed(0)
best_knockouts, best_reward = ga_embeds.run()
print("Best knockouts:", best_knockouts)
print("Best reward:", best_reward)

  0%|          | 1/300 [00:03<19:46,  3.97s/it]

Epoch: 0, Loss: 10561118208.0


 17%|█▋        | 51/300 [02:50<14:03,  3.39s/it]

Epoch: 50, Loss: 2933362432.0


 34%|███▎      | 101/300 [05:36<10:39,  3.21s/it]

...

Epoch: 250, Loss: 5796399.5


100%|██████████| 300/300 [16:41<00:00,  3.34s/it]

Epoch: 299, Loss: 5649187.5



 14%|█▎        | 27/200 [04:57<33:38, 11.67s/it]

: 